# Dunnhumby seed43 — 공유 특징 임베딩 + 개별 ID 편차 M2
**새 학습은 M2 하나입니다.** 기존 seed43 M1의 전체 곡선과 같은 조기종료 규칙을 재사용합니다. 기준 결과가 없거나 다르면 중단하고, M1·M3·M4·M5·H&M·추가 시드를 자동 학습하지 않습니다.

사용자 layer0 = ID + W_N·φ(q_N) + W_V·φ(q_V), 상품 layer0 = ID + W_P·φ(전체 가격 백분위) + W_PC·φ(카테고리 내 가격 백분위). φ는 기존 3차원 RBF에서 **유효 학습 노드 평균만 뺀 값**입니다. N과 V를 모두 유지하며 N은 거래 활동이지 동일상품 재구매율이 아닙니다. 기존 상품 구매자 N 평균은 사용하지 않습니다. q_C를 따로 넣지 않는 **historical CLV 구성요소 M2**입니다.

네 공유행렬은 0에서 시작하고, ID는 M1과 같은 순서로 초기화합니다. 합성된 layer0에서 동일한 binary LightGCN 2층을 전파하며, 하나의 optimizer·plain BPR로 전부 공동학습합니다. 동결·보조손실·별도 점수 rho·외부 재정렬은 없습니다. 추가 파라미터는 768개이며 전파 차원은64로 유지합니다.

규제 = .001×사용자 ID 제곱합/B + .001×양성·음성 상품 ID 제곱합/B + .001×네 공유행렬 제곱합. 공유행렬을 B로 나누지 않고, 합성 표현이 아닌 **개별 ID와 공유행렬을 따로 규제**합니다. 사용자/아이템/축별 계수는 후속 데이터별 조정이 가능하지만 이번에는 전부 .001 하나만 실험합니다.

고정: Dunnhumby seed43·ID64·2층·lr=.0005·batch8192·uniform unseen K=1·MIN_ITEM_INTER=1. 학습≤683 / 반복 노출 개발684–690, 신규상품 추천(학습 user-item 제외). 최종 test·holdout 없음. 기존 출력의 'test'는 개발684–690의 구형 명칭입니다.

**최대300 epoch, 25마다 평가, 전체 가격·구매금액 가중 적중값@10으로 선택합니다.** 동률은 이른 epoch, 100 이후 4회 미개선이면 중단하고 최적값을 복원합니다. M1 재사용 결과는 선택100/종료200입니다. 다른 모델이 다른 epoch에서 종료해도 동일 규칙의 독립 선택으로 비교하고, 같은 epoch 비교는 관측된 교집합만 별도 저장합니다.

판정: 전체 경제2지표@10이 M1 초과, 전체 Recall/NDCG @10/20/50 각각 M1의99% 이상. 전체·저/중/고CLV·노출·불리한 결과도 보존합니다. 단일 개발시드로 유의성·일반화·CLV 귀속을 주장하지 않습니다. 초기 gradient, 이후 N/V 표현, 선택 checkpoint의 N/V 제거별 정답 진입·이탈을 기록합니다. **제거 진단의 ID는 이미 새 M2에서 학습된 것이므로 M1이 아닙니다.**

로컬 검증: 합성2epoch 실제 평가·저장·중단/재개·완료 재사용 통과, 초기 M1 점수 및 L2 수식 확인. 실제 Dunnhumby 데이터 준비와 초기 네 축 BPR gradient 확인. 전체 GPU 학습 및 Colab Drive 연결은 아직 수행하지 않았습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_nv_modulated_id_seed43_v3/reports/result.json'
# 이전 JSON을 옮겼다면 REPORT만 수정하세요. nv_modulated_id_m2_m5_seed43_results.zip도 허용됩니다.
try:
    if not ROOT.is_dir():
        drive.mount('/content/drive')
    if not ROOT.is_dir():
        raise RuntimeError('논문/data 폴더가 없습니다. Drive 계정과 ROOT 경로를 확인하세요.')
    if not REPORT.is_file():
        raise RuntimeError(f'기준 원본 없음: {REPORT}. REPORT를 기존 JSON/ZIP 경로로 수정하세요. 학습은 시작하지 않았습니다.')
except (OSError, ValueError, NotImplementedError) as exc:
    raise RuntimeError('Drive 연결/읽기 실패. 마운트 상태를 확인하고 이 셀부터 재실행하세요. 학습은 시작하지 않았습니다.') from exc
SOURCE_COMMIT = 'b739215e83b1c59b7a406668877fb927960be733'
REPO = Path('/content/clv-shared-feature-residual-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '다른 실험 코드가 로드되어 있습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_shared_feature_residual_m2_screen as screen
import pandas as pd
OUT = ROOT/'results_v3_dunnhumby_shared_feature_residual_m2_seed43_v1'
assert screen.VERSION == 'shared-feature-residual-m2-seed43-development-v1'
screen.read_source(REPORT, screen.configure(str(OUT)))
print('기존 M1 결과·학습 규칙 및 고정 소스 확인:', SOURCE_COMMIT)

## 1. 입력·기준 결과 확인 — 학습 없음
N/V 유효 마스크와 가격 특징의 유효 행 수를 출력합니다. 데이터와 원본 M1의 학습 규칙이 다르면 학습 전에 중단합니다.

In [ ]:
cfg, prepared = screen.prepare(REPORT, OUT)
assert cfg.seeds == (43,) and cfg.epochs == 300 and cfg.pref_reg == .001
assert [a['model_id'] for a in prepared['anchors']] == ['m1']
assert screen.spec()['role'] == 'M2' and not screen.spec()['weighted']
print('새 학습:', screen.spec())
print('규제:', prepared['feature_settings'])
print('선택 규칙:', screen.selection(cfg))

## 2. 새 M2 하나 학습
완료 epoch마다 optimizer·난수상태를 Drive에 저장합니다. 중단 후 같은 노트북을 처음부터 재실행하면 이어집니다. 완료 모델은 추가 학습하지 않습니다. 초기 N/V gradient나 이후 N/V 표현이 수치0이면 진단을 남기고 중단합니다.

학습 후 선택 checkpoint만 사용해 N/V 제거 진단을 수행합니다. **추가 모델 학습이 아니라 동일 모델의 작동 확인**이며, 제거한 설정을 추천모형으로 선택하지 않습니다. 단일 CPU 예제 통과는 성능 개선 보장이 아닙니다.

In [ ]:
import torch
assert torch.cuda.is_available(), '전체 실험에는 GPU 런타임을 사용하세요.'
paths = screen.run(cfg, prepared)
print(json.dumps(paths, ensure_ascii=False, indent=2))

## 3. 전체 결과·학습곡선·ZIP
생략 없는 전체 지표와 진단을 함께 판독합니다. ZIP에는 checkpoint 자체를 넣지 않아 다운로드 용량을 줄이며, checkpoint/optimizer는 Drive에 그대로 남습니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
import matplotlib.pyplot as plt

report_path = OUT/'reports/result.json'
paths = {}
if report_path.is_file():
    report = json.loads(report_path.read_text())
    assert report['code_version'] == screen.VERSION
    paths = report['paths']
    absolute = pd.read_csv(paths['absolute'])
    print('전체 절대지표 — 기존 M1, 새 공유 특징+개별 ID M2')
    print(absolute.set_index('model_id').T.to_string())
    print(json.dumps(report['reading'], ensure_ascii=False, indent=2))
    print('독립 선택 비교:', paths['comparison'])
    print('같은 epoch 비교:', paths['same_epoch_comparison'])
    print('N/V 작동:', paths['diagnostics'])
    print('N/V 제거별 정답 진입/이탈:', paths['movement_summary'])
    curve = pd.read_csv(paths['curve'])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, title in zip(axes,
            ['recall@10', 'price_purchase_amount_weighted_hit@10'],
            ['Development Recall@10', 'Development weighted hit@10']):
        for mid, part in curve.groupby('model_id', sort=False):
            ax.plot(part.epoch, part[metric], marker='o', label=mid)
        ax.axvline(100, color='grey', linestyle='--')
        ax.set(xlabel='epoch', title=title)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT/'reports/learning_curve.png', dpi=150)
    plt.show()
else:
    print('완료 결과 없음. 저장된 초기/중간 진단만 내려받습니다. 성공/실패 판정은 보류합니다.')

zip_path = Path('/content/shared_feature_residual_m2_seed43_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
    for path in (OUT/'feature_diagnostic.json', OUT/'reports/learning_curve.png'):
        if path.is_file():
            archive.write(path, arcname=path.name)
    for path in OUT.glob('arms/*/probe_epoch*.json'):
        archive.write(path, arcname=str(path.relative_to(OUT)))
files.download(str(zip_path))